In [ ]:
!pip install transformers torch scikit-learn tqdm

In [ ]:
import json
import torch
import numpy as np

from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModel
)

In [ ]:
MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModel.from_pretrained(
    MODEL_NAME
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json(
    "/content/train.json"
)

test_data = load_json(
    "/content/test.json"
)

print(len(train_data))
print(len(test_data))

5798
1450


In [ ]:
print(train_data[0])

{'ID': 'vumtps', 'Comment': 'आज जीउदै छौ कसलाई के थाहा छ र भोली जीउदै भईन्छ की भईदैन मरीलानू के छ र एक जूनीमा😃सकेसम्म राम्रो बन्न सीक😃सीके सीक नसीके नसीक😉😋😅😂', 'Label_Binary': 'NOFF', 'Label_Multiclass': 'NO'}


In [ ]:
train_texts = [
    row["Comment"]
    for row in train_data
]

test_texts = [
    row["Comment"]
    for row in test_data
]

train_binary = [
    row["Label_Binary"]
    for row in train_data
]

test_binary = [
    row["Label_Binary"]
    for row in test_data
]

In [ ]:
train_multi = [
    row["Label_Multiclass"]
    for row in train_data
]

test_multi = [
    row["Label_Multiclass"]
    for row in test_data
]

In [ ]:
binary_map = {
    "NOFF": 0,
    "OFF": 1
}

y_train_binary = np.array([
    binary_map[x]
    for x in train_binary
])

y_test_binary = np.array([
    binary_map[x]
    for x in test_binary
])

In [ ]:
multi_map = {
    "NO": 0,   # non offensive
    "OO": 1,   # offensive general
    "OR": 2,   # offensive targeted/racist
    "OS": 3    # offensive sexist/other
}

In [ ]:
y_train_multi = np.array([
    multi_map[x]
    for x in train_multi
])

y_test_multi = np.array([
    multi_map[x]
    for x in test_multi
])

In [ ]:
def get_embedding(
    text,
    max_len=128
):
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    with torch.no_grad():

        outputs = model(
            **encoded
        )

    cls_embedding = (
        outputs.last_hidden_state[
            :, 0, :
        ]
        .squeeze()
        .numpy()
    )

    return cls_embedding

In [ ]:
X_train = []

for text in tqdm(train_texts):
    emb = get_embedding(text)
    X_train.append(emb)

X_train = np.array(
    X_train
)

100%|██████████| 5798/5798 [31:44<00:00,  3.04it/s]


In [ ]:
X_test = []

for text in tqdm(test_texts):
    emb = get_embedding(text)
    X_test.append(emb)

X_test = np.array(
    X_test
)

100%|██████████| 1450/1450 [08:50<00:00,  2.73it/s]


In [ ]:
print(X_train.shape)
print(X_test.shape)

(5798, 768)
(1450, 768)


In [ ]:
np.save(
    "X_train_mbert.npy",
    X_train
)

np.save(
    "X_test_mbert.npy",
    X_test
)

np.save(
    "y_train_binary.npy",
    y_train_binary
)

np.save(
    "y_test_binary.npy",
    y_test_binary
)

To save compute time , store this

In [ ]:
X_train = np.load(
    "X_train_mbert.npy"
)

In [ ]:
X_test = np.load(
    "X_test_mbert.npy"
)